In [26]:
from torch.utils.data import Dataset
import torch
import random
import torch.nn as nn

In [5]:
with open("text.txt", "r") as f:
  text = f.read()

#print(text)

Kot siedzi na parapecie i obserwuje ptaki za oknem. Pies biega po ogrodzie i goni kolorową piłkę. Dziecko czyta książkę przed snem i uczy się nowych słów. Nauczyciel tłumaczy uczniom podstawy matematyki oraz fizyki. Student pisze program w języku Python i testuje kolejne funkcje. Komputer przetwarza dane szybciej niż człowiek wykonuje obliczenia. Sieć neuronowa analizuje tekst i przewiduje następne słowo. Model językowy uczy się zależności pomiędzy wyrazami na podstawie dużego zbioru danych. Algorytm optymalizuje parametry podczas kolejnych epok treningu. Funkcja kosztu zmniejsza się wraz z postępem uczenia. Samochód jedzie drogą przez las i mija małą wioskę. Rowerzysta zatrzymuje się przed przejściem dla pieszych. Pociąg przyjeżdża na stację zgodnie z rozkładem jazdy. Samolot startuje z lotniska i kieruje się na południe. Wieczorem ludzie spacerują po parku i rozmawiają o codziennych sprawach. Rano słońce oświetla miasto i budzi mieszkańców. Deszcz pada przez kilka godzin, a wiatr por

In [34]:
class SkipGramDataset(Dataset):
  def __init__(self, text, k=5):
    self.pairs = []

    words = text.split()

    self.vocab = sorted(set(words))
    text_to_idx = {x : i for i,x in enumerate(self.vocab)}

    #print(text_to_idx)

    window = 2

    for i in range(len(words)):
      start = max(0, i-window)
      end = min(len(words), i+window+1)

      context = words[start:i] + words[i+1:end]
      negative = set()

      while len(negative) < k:
        losowa = random.randint(0, len(self.vocab)-1)

        if self.vocab[losowa] in context or self.vocab[losowa] == words[i]:
          continue

        negative.add(text_to_idx[self.vocab[losowa]])

      for word in context:
        self.pairs.append((text_to_idx[words[i]], text_to_idx[word], list(negative)))

    #print(self.pairs)

  def __getitem__(self, idx):
    return self.pairs[idx]

  def __len__(self):
    return len(self.pairs)

dataset = SkipGramDataset(text)

In [33]:
class SkipGramNSModel(nn.Module):
  def __init__(self, vocab_size, embedding_dim):
    super().__init__()
    self.in_embedding = nn.Embedding(vocab_size, embedding_dim)
    self.out_embedding = nn.Embedding(vocab_size, embedding_dim)

  def forward(self, center, pos_samples, neg_samples):
    v = self.in_embedding(center) # [batch_size, embed_dim]
    u_pos = self.out_embedding(pos_samples) # [batch_size, embed_dim]
    u_neg = self.out_embedding(neg_samples) # [batch_size, k, embed_dim]

    pos_score = torch.sum(u_pos * v, dim=1)
    neg_score = torch.bmm(u_neg, v.unsqueeze(2)).squeeze(2)

    return (pos_score, neg_score)

  def loss(self, pos_score, neg_score):
    return -(torch.log(torch.sigmoid(pos_score)) + torch.sum(torch.log(torch.sigmoid(-neg_score)),dim=1)).mean()